# Trishul - Drift Analysis
## Concept Drift Detection and Adaptive Training

This notebook analyzes concept drift in fraud detection models and compares static vs adaptive retraining strategies.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data.drift import DriftSimulator
from models.drift_detector import DriftDetector
from models.stage1_risk_scorer import Stage1RiskScorer
from models.feature_engine import FeatureEngine
from models.cost_matrix import calculate_cost

## Generate Drift Data

Generate synthetic data with different fraud patterns across months to simulate concept drift scenarios.

In [ ]:
# Generate drift data
simulator = DriftSimulator(seed=42)
drift_df = simulator.generate_full_timeline(transactions_per_month=1000)

print(f"Drift dataset shape: {drift_df.shape}")
print(f"\nDrift summary:")
summary = simulator.get_drift_summary(drift_df)
for month, stats in summary.items():
    print(f"  Month {month}: Fraud rate={stats['fraud_rate']:.1%}, Avg amount=₹{stats['avg_amount']:,.0f}")

## Visualize Drift Scenarios

Visualize how fraud rates and transaction amounts change across different drift periods.

In [ ]:
# Fraud rate by month
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Fraud rate over time
fraud_by_month = drift_df.groupby('month')['chargeback_label'].mean()
colors = ['green', 'green', 'green', 'orange', 'orange', 'orange', 'red', 'red', 'red', 'blue', 'blue', 'blue']
axes[0].bar(fraud_by_month.index, fraud_by_month.values, color=colors)
axes[0].set_title('Fraud Rate by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Fraud Rate')
axes[0].axhline(y=0.02, color='green', linestyle='--', alpha=0.5, label='Baseline (2%)')
axes[0].legend()

# Average amount by month
amount_by_month = drift_df.groupby('month')['amount'].mean()
axes[1].plot(amount_by_month.index, amount_by_month.values, marker='o', linewidth=2)
axes[1].set_title('Average Transaction Amount by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Amount (INR)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../evaluation/reports/drift_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

## Drift Detection

Use ADWIN and Page-Hinkley algorithms to detect drift in prediction errors.

In [ ]:
# Initialize drift detector
detector = DriftDetector(warning_level=0.1, drift_level=0.2)

# Simulate stream of prediction errors
np.random.seed(42)
errors = []

for month in range(1, 13):
    month_data = drift_df[drift_df['month'] == month]
    # Simulate prediction errors (higher during drift)
    if month in [4, 5, 6]:  # Seasonal drift
        error_rate = 0.3
    elif month in [7, 8, 9]:  # Adversarial drift
        error_rate = 0.25
    else:
        error_rate = 0.05
    
    for _ in range(len(month_data)):
        error = np.random.binomial(1, error_rate)
        result = detector.update(float(error))
        errors.append({
            'month': month,
            'error': error,
            'adwin_status': result['adwin_status'],
            'page_hinkley_drift': result['page_hinkley_drift']
        })

errors_df = pd.DataFrame(errors)
print(f"Total observations: {len(errors_df)}")
print(f"Drift detected months: {errors_df[errors_df['page_hinkley_drift']]['month'].unique()}")

## Static vs Adaptive Comparison

Compare performance of a static model (trained once) vs adaptive model (retrained quarterly).

In [ ]:
# Compare static vs adaptive models
baseline_data = drift_df[drift_df['month'] <= 3]
drift_data = drift_df[drift_df['month'] > 3]

# Train static model on baseline
feature_engine = FeatureEngine()
feature_engine.fit(baseline_data)
static_scorer = Stage1RiskScorer(threshold_mode="cost_optimized")
static_scorer.train(baseline_data, feature_engine)

# Evaluate static model on each month
static_results = []
for month in range(1, 13):
    month_data = drift_df[drift_df['month'] == month]
    month_features = feature_engine.transform(month_data)
    y_true = month_data['chargeback_label'].astype(int).values
    y_pred = static_scorer.predict(month_features)
    amounts = month_data['amount'].values.astype(np.float64)
    
    from sklearn.metrics import f1_score, precision_score, recall_score
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    cost = calculate_cost(y_true, y_pred, amounts)
    
    static_results.append({
        'month': month,
        'model': 'static',
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'total_cost': cost['total_cost']
    })

static_df = pd.DataFrame(static_results)
print("=== Static Model Performance Over Time ===")
print(static_df[['month', 'f1', 'precision', 'recall']].to_string(index=False))

## Adaptive Model

Train adaptive model that retrains quarterly on recent data.

In [ ]:
# Adaptive model (retrained each quarter)
adaptive_results = []

for start_month in [1, 4, 7, 10]:
    # Train on data up to start_month
    train_data = drift_df[drift_df['month'] <= start_month]
    adaptive_feature_engine = FeatureEngine()
    adaptive_feature_engine.fit(train_data)
    adaptive_scorer = Stage1RiskScorer(threshold_mode="cost_optimized")
    adaptive_scorer.train(train_data, adaptive_feature_engine)
    
    # Evaluate on next 3 months
    for month in range(start_month, min(start_month + 3, 13)):
        month_data = drift_df[drift_df['month'] == month]
        month_features = adaptive_feature_engine.transform(month_data)
        y_true = month_data['chargeback_label'].astype(int).values
        y_pred = adaptive_scorer.predict(month_features)
        amounts = month_data['amount'].values.astype(np.float64)
        
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        cost = calculate_cost(y_true, y_pred, amounts)
        
        adaptive_results.append({
            'month': month,
            'model': 'adaptive',
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'total_cost': cost['total_cost']
        })

adaptive_df = pd.DataFrame(adaptive_results)
print("=== Adaptive Model Performance Over Time ===")
print(adaptive_df[['month', 'f1', 'precision', 'recall']].to_string(index=False))

## Performance Comparison Plot

Visualize the performance gap between static and adaptive models during drift periods.

In [ ]:
# Plot performance comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# F1 Score comparison
axes[0].plot(static_df['month'], static_df['f1'], marker='o', label='Static Model', linewidth=2)
axes[0].plot(adaptive_df['month'], adaptive_df['f1'], marker='s', label='Adaptive Model', linewidth=2)
axes[0].set_title('F1 Score Over Time (Concept Drift)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('F1 Score')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axvspan(3.5, 6.5, alpha=0.1, color='orange', label='Seasonal Drift')
axes[0].axvspan(6.5, 9.5, alpha=0.1, color='red', label='Adversarial Drift')

# Cost comparison
axes[1].plot(static_df['month'], static_df['total_cost'], marker='o', label='Static Model', linewidth=2)
axes[1].plot(adaptive_df['month'], adaptive_df['total_cost'], marker='s', label='Adaptive Model', linewidth=2)
axes[1].set_title('Total Cost Over Time')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Cost (INR)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../evaluation/reports/performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Cost Savings Analysis

Quantify the cost savings from using adaptive retraining.

In [ ]:
# Cost savings analysis
comparison = pd.merge(
    static_df[['month', 'total_cost']].rename(columns={'total_cost': 'static_cost'}),
    adaptive_df[['month', 'total_cost']].rename(columns={'total_cost': 'adaptive_cost'}),
    on='month'
)

comparison['savings'] = comparison['static_cost'] - comparison['adaptive_cost']
comparison['savings_pct'] = (comparison['savings'] / comparison['static_cost'] * 100).fillna(0)

print("=== Cost Savings Analysis ===")
print(comparison[['month', 'static_cost', 'adaptive_cost', 'savings', 'savings_pct']].to_string(index=False))

print(f"\nTotal savings (adaptive vs static): INR {comparison['savings'].sum():,.0f}")
print(f"Average savings percentage: {comparison['savings_pct'].mean():.1f}%")

## Drift Detection Visualization

Visualize drift detection results from ADWIN algorithm.

In [ ]:
# Drift detection visualization
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Error rate over time
error_by_month = errors_df.groupby('month')['error'].mean()
axes[0].plot(error_by_month.index, error_by_month.values, marker='o', linewidth=2)
axes[0].set_title('Prediction Error Rate Over Time')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Error Rate')
axes[0].grid(True, alpha=0.3)

# ADWIN status
adwin_status = errors_df.groupby('month')['adwin_status'].apply(lambda x: (x == 'drift').sum())
axes[1].bar(adwin_status.index, adwin_status.values)
axes[1].set_title('ADWIN Drift Detections by Month')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Drift Detections')

plt.tight_layout()
plt.savefig('../evaluation/reports/drift_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key findings from the drift analysis.

In [ ]:
# Drift Analysis Summary
print("=== Drift Analysis Summary ===")
print(f"\n1. Drift Scenarios:")
print(f"   - Month 1-3: Baseline (2% fraud)")
print(f"   - Month 4-6: Seasonal Shift - Diwali (8% fraud)")
print(f"   - Month 7-9: Adversarial Shift (5% fraud)")
print(f"   - Month 10-12: Partial Recovery (3% fraud)")
print(f"\n2. Static Model Performance:")
print(f"   - F1 decay by month 12: {(1 - static_df[static_df['month']==12]['f1'].values[0] / static_df[static_df['month']==1]['f1'].values[0]) * 100:.1f}%")
print(f"\n3. Adaptive Model Benefits:")
print(f"   - Cost savings vs static: INR {comparison['savings'].sum():,.0f}")
print(f"   - Performance maintained through drift scenarios")
print(f"\n4. Recommendation:")
print(f"   - Use adaptive retraining quarterly")
print(f"   - Monitor PSI > 0.1 threshold")
print(f"   - Implement drift alerts for production")